In [2]:
import json
from pathlib import Path

import numpy as np

from pyscf import gto

import sys

sys.path.append("../cadft")
from cc2cc.utils.basis import gen_basis

BASIS = "cc-pVDZ"

In [2]:
dict_ = {"molecule": [], "spin": {}, "charge": {}}

data_path = Path("./sets")
# download the data from https://github.com/obackhouse/gmtkn
# Note
#   1) The data is not included in this repository.
#   2) The data included GMTKN55, GW100, GAPS, MRADC, ACC24, S30L, BH9.
max_num_atom = -1

for python_path in data_path.glob("*.py"):
    if python_path.name == "__init__.py":
        continue
    exec(python_path.read_text())
    # load the systems and reactions from the executed python file
    dataset = python_path.stem
    print(f"Processing {dataset}")
    dict_[f"molecule_{dataset}"] = []

    for i_name in systems:
        molecule = []
        for i_atom in range(len(systems[i_name]["atoms"])):
            molecule.append(
                [
                    systems[i_name]["atoms"][i_atom],
                    float(systems[i_name]["coords"][i_atom][0]),
                    float(systems[i_name]["coords"][i_atom][1]),
                    float(systems[i_name]["coords"][i_atom][2]),
                ]
            )

        try:
            mol = gto.M(
                atom=molecule,
                basis=gen_basis(molecule, BASIS, True),
                verbose=0,
                charge=systems[i_name]["charge"],
                spin=systems[i_name]["spin"],
                unit="B",
            )
            mol.build()
            max_num_atom = max(max_num_atom, len(molecule))
        except Exception as e:
            print(molecule)
            print(f"Error in {i_name}: {e}")
            continue

        molecule = []
        for i_atom in mol._atom:
            molecule.append(
                [
                    i_atom[0],
                    i_atom[1][0],
                    i_atom[1][1],
                    i_atom[1][2],
                ]
            )

        dict_i_name = f"{dataset}-{i_name}"
        dict_[dict_i_name] = molecule
        dict_[f"molecule_{dataset}"].append(dict_i_name)
        dict_["molecule"].append(dict_i_name)
        dict_["charge"][dict_i_name] = systems[i_name]["charge"]
        dict_["spin"][dict_i_name] = systems[i_name]["spin"]

    dict_[f"reaction-{dataset}"] = {}
    for i_reaction, reaction in enumerate(reactions):
        dict_[f"reaction-{dataset}"][i_reaction] = {
            "systems": reaction["systems"],
            "stoichiometry": reaction["stoichiometry"],
            "reference": reaction["reference"],
        }

    for number_atom in range(max_num_atom):
        dict_[f"molecule{number_atom}-{dataset}"] = []

    for molecule in dict_[f"molecule_{dataset}"]:
        num_heavy_atom = 0
        num_atom = 0
        for atom in dict_[molecule]:
            num_atom += 1
            if atom[0].upper() != "H":
                num_heavy_atom += 1

        if num_heavy_atom == 0:
            if num_atom == 1:
                dict_[f"molecule0-{dataset}"].append(molecule)
            else:
                dict_[f"molecule1-{dataset}"].append(molecule)
        elif num_heavy_atom == 1:
            if num_atom == 1:
                dict_[f"molecule0-{dataset}"].append(molecule)
            else:
                dict_[f"molecule1-{dataset}"].append(molecule)
        else:
            dict_[f"molecule{num_heavy_atom}-{dataset}"].append(molecule)

for number_atom in range(max_num_atom):
    dict_[f"molecule{number_atom}"] = []

for molecule in dict_["molecule"]:
    num_heavy_atom = 0
    num_atom = 0
    for atom in dict_[molecule]:
        num_atom += 1
        if atom[0].upper() != "H":
            num_heavy_atom += 1

    if num_heavy_atom == 0:
        if num_atom == 1:
            dict_[f"molecule0"].append(molecule)
        else:
            dict_[f"molecule1"].append(molecule)
    elif num_heavy_atom == 1:
        if num_atom == 1:
            dict_[f"molecule0"].append(molecule)
        else:
            dict_[f"molecule1"].append(molecule)
    else:
        dict_[f"molecule{num_heavy_atom}"].append(molecule)
print()
print("DONE")

Processing BHPERI
Processing RG18
Processing AHB21
Processing ICONF
Processing HAL59
[['I', 0.003414377, -3.215655817, 1.72846281], ['C', 0.106167253, -5.333396952, 1.850723289], ['F', 0.135277584, -5.877100968, 0.636074544], ['F', 1.202218878, -5.716996848, 2.501577262], ['F', -0.950096543, -5.821924316, 2.496892131]]
Error in 30_CF3I-benB: 'Element i (Z=53) not found in basis cc-pvdz version 1'
[['C', -0.755422531, -0.796459123, -1.023590391], ['C', 0.634274834, -0.880017014, -1.075233285], ['C', 1.406955202, 0.199695367, -0.653144334], ['C', 0.798863737, 1.361204515, -0.180597909], ['C', -0.593166787, 1.434312023, -0.133597923], ['C', -1.376239198, 0.359205222, -0.553258516], ['I', -1.514344238, 3.173268101, 0.573601106], ['H', 1.110906949, -1.778801728, -1.440619836], ['H', 1.399172302, 2.197767355, 0.147412751], ['H', 2.48641778, 0.142466525, -0.689380574], ['H', -2.45425225, 0.42258112, -0.512807958], ['H', -1.362353593, -1.630564523, -1.348743149], ['S', -3.112683203, 6.28922783

In [ ]:
# import json

# mol_name = [
#     "molecule_W4_11",
#     "molecule_G21EA",
#     "molecule_G21IP",
#     "molecule_DIPCS10",
#     "molecule_PA26",
#     "molecule_SIE4x4",
#     "molecule_ALKBDE10",
#     "molecule_YBDE18",
#     "molecule_AL2X6",
#     "molecule_HEAVYSB11",
#     "molecule_NBPRC",
#     "molecule_ALK8",
#     "molecule_RC21",
#     "molecule_G2RC",
#     "molecule_BH76",
#     "molecule_FH51",
#     "molecule_TAUT15",
#     "molecule_DC13",
#     "molecule_MB16_43",
#     "molecule_DARC",
#     "molecule_RSE43",
#     "molecule_BSR36",
#     "molecule_CDIE20",
#     "molecule_ISO34",
#     "molecule_ISOL24",
#     "molecule_C60ISO",
#     "molecule_PArel",
#     "molecule_BHPERI",
#     "molecule_BHDIV10",
#     "molecule_INV24",
#     "molecule_BHROT27",
#     "molecule_PX13",
#     "molecule_WCPT18",
#     "molecule_RG18",
#     "molecule_ADIM6",
#     "molecule_S22",
#     "molecule_S66",
#     "molecule_HEAVY28",
#     "molecule_WATER27",
#     "molecule_CARBHB12",
#     "molecule_PNICO23",
#     "molecule_HAL59",
#     "molecule_AHB21",
#     "molecule_CHB6",
#     "molecule_IL16",
#     "molecule_IDISP",
#     "molecule_ICONF",
#     "molecule_ACONF",
#     "molecule_Amino20x4",
#     "molecule_PCONF21",
#     "molecule_MCONF",
#     "molecule_SCONF",
#     "molecule_UPU23",
#     "molecule_BUT14DIOL",
# ]

# data_set = "gmtkn-cc-pVDZ"
# with open(f"../cadft/utils/{data_set}.json") as f:
#     json_data = json.load(f)

# len_mol = 0
# for i in mol_name:
#     len_mol += len(json_data[i])
#     # print(i, len(json_data[i]))
# print(len_mol)

# len_mol = 0
# for i in range(10):
#     len_mol += len(json_data[f"molecule{i}"])
# print(len_mol)


2375
2774


In [3]:
dict_new = {}
for key in dict_:
    if key.startswith("molecule"):
        if dict_[key]:
            dict_new[key] = dict_[key]
    else:
        dict_new[key] = dict_[key]

with open(f"gmtkn-{BASIS}.json", "w") as f:
    json.dump(dict_new, f)

In [3]:
# for clean up
import json
import re
import sys

import numpy as np


sys.path.append("../cadft")
from cc2cc.utils.basis import gen_basis
from cc2cc.utils.rotate import rotate

dict_new = {"molecule": [], "spin": {}, "charge": {}}

with open(f"gmtkn-{BASIS}.json", "r") as f:
    dict_ = json.load(f)

for key in dict_:
    if re.match(r"molecule[0-9]+", key):
        dict_new[key] = []
        print(f"Processing {dict_[key]}")
        for molecule in dict_[key]:
            molecule_list = np.array(dict_[molecule], dtype=object)
            molecule_list = molecule_list[np.argsort(molecule_list[:, 0])]
            rotate(molecule_list)

            add_flag = True

            duplicate_flag = False
            for molecule_new in dict_new[key]:
                molecule_new_list = np.array(dict_[molecule_new], dtype=object)
                molecule_new_list = molecule_new_list[
                    np.argsort(molecule_new_list[:, 0])
                ]
                rotate(molecule_new_list)

                if len(molecule_new_list) != len(molecule_list):
                    continue

                for i_atom in range(len(molecule_list)):
                    if (
                        molecule_new_list[i_atom][0] == molecule_list[i_atom][0]
                        and (
                            dict_["charge"][molecule] == dict_["charge"][molecule_new]
                            and dict_["spin"][molecule] == dict_["spin"][molecule_new]
                        )
                        and np.linalg.norm(
                            molecule_new_list[i_atom][1:] - molecule_list[i_atom][1:]
                        )
                        < 1e-7
                    ):
                        duplicate_flag = True
                    else:
                        duplicate_flag = False
                        break

                if duplicate_flag:
                    add_flag = False
                    break

            if add_flag:
                dict_new[key].append(molecule)
                dict_new[molecule] = dict_[molecule]
                dict_new["molecule"].append(molecule)
                dict_new["charge"][molecule] = dict_["charge"][molecule]
                dict_new["spin"][molecule] = dict_["spin"][molecule]
            else:
                print(
                    f"spin and charge: [{dict_['spin'][molecule]}, {dict_['charge'][molecule]}] with [{dict_['spin'][molecule_new]}, {dict_['charge'][molecule_new]}]"
                )
                print(
                    f"Comparing atom and coordinates: \n {molecule_new_list} \n and \n {molecule_list}"
                )
                print(f"Duplicate found: {molecule} and {molecule_new}\n")
                dict_new[molecule] = molecule_new

        print(len(dict_[key]), len(dict_new[key]))
        print(f"remove duplicates: {[_ for _ in dict_[key] if _ not in dict_new[key]]}")
        print()
    elif re.match(r"reaction-.*", key):
        dict_new[key] = dict_[key]
        print(f"Reaction '{key}'.")
    elif re.match(r"molecule_.*", key):
        dict_new[key] = dict_[key]
        print(f"molecule key '{key}' processed.")

with open(f"filtered_gmtkn-{BASIS}.json", "w") as f:
    json.dump(dict_new, f)

print(f"Filtered data has been saved to filtered_gmtkn-{BASIS}.json")
print("DONE")

molecule key 'molecule_ADDON' processed.
Processing ['ADDON_Se', 'ADDON_Ge', 'ADDON_As']
3 3
remove duplicates: []

molecule key 'molecule_BHPERI' processed.
Reaction 'reaction-BHPERI'.
Processing ['BHPERI-13_c2h4', 'BHPERI-Ethylene', 'BHPERI-00r']
3 3
remove duplicates: []

Processing ['BHPERI-13r_6', 'BHPERI-13r_1', 'BHPERI-13r_5', 'BHPERI-13r_8', 'BHPERI-13r_9', 'BHPERI-13r_4', 'BHPERI-13r_3', 'BHPERI-13r_7', 'BHPERI-13r_2']
9 9
remove duplicates: []

Processing ['BHPERI-Cyclobutene', 'BHPERI-1,3-Butadiene', 'BHPERI-TS1', 'BHPERI-01r']
4 4
remove duplicates: []

Processing ['BHPERI-1,3-Pentadiene', 'BHPERI-1,3-Cyclopentadiene', 'BHPERI-13ts_4a', 'BHPERI-13ts_3a', 'BHPERI-13ts_9a', 'BHPERI-13ts_5a', 'BHPERI-13ts_7a', 'BHPERI-05r', 'BHPERI-04r', 'BHPERI-13ts_6a', 'BHPERI-02r', 'BHPERI-TS5', 'BHPERI-13ts_2a', 'BHPERI-TS4', 'BHPERI-03r', 'BHPERI-13ts_1a', 'BHPERI-09r', 'BHPERI-07r', 'BHPERI-13ts_8a']
19 19
remove duplicates: []

Processing ['BHPERI-TS7', 'BHPERI-cis-1,3,5-Hexatriene', '